Import Python modules

In [1]:
import os
import glob
import bte
from collections import defaultdict
from collections import Counter
import pandas as pd
import numpy as np
import math

import sys
sys.path.append('../scripts/')
import MATWrapper
import ExpectedCalc

resultsdir = '../results/'
if not os.path.isdir(resultsdir):
    os.makedirs(resultsdir)

Read in trees

In [2]:
# data_dirs = {
#     'H3N2' : 'A_New_York_392_2004_H3N2',
#     'H1N1' : 'A_Puerto_Rico_8_1934_H1N1',
#     'H1N1pdm' : 'A_California_07_2009_H1N1',
#     'H7N9' : 'A_Shanghai_02_2013_H7N9',
#     'H5N1' : 'A_goose_Guangdong_1_1996_H5N1',
#     'H2N2' : 'A_Korea_426_1968_H2N2',
#     'H9N2' : 'A_Hong_Kong_1073_99_H9N2'
# }

subtypes = ['H1N1', 'H3N2', 'H5N1', 'H7N9', 'H9N2']
segments = ['PB2', 'PB1', 'PA', 'HA', 'NP', 'NA', 'NS', 'MP']
trees = {}
for subtype in subtypes:
    trees[subtype] = {}
    for segment in segments:
        tree_file = f'../../flu-usher/results/{subtype}/{segment}/opt_tree.pb.gz'
        if not os.path.isfile(tree_file):
            continue
        tree = bte.MATree(tree_file)
        trees[subtype][segment] = tree

Finished 'from_pb' in 0.9517 seconds
Finished 'from_pb' in 1.1043 seconds
Finished 'from_pb' in 0.7125 seconds
Finished 'from_pb' in 0.9656 seconds
Finished 'from_pb' in 0.7808 seconds
Finished 'from_pb' in 0.9135 seconds
Finished 'from_pb' in 1.0204 seconds
Finished 'from_pb' in 0.9662 seconds
Finished 'from_pb' in 0.8371 seconds
Finished 'from_pb' in 0.8578 seconds
Finished 'from_pb' in 0.8795 seconds
Finished 'from_pb' in 1.1812 seconds
Finished 'from_pb' in 0.8921 seconds
Finished 'from_pb' in 1.0754 seconds
Finished 'from_pb' in 0.833 seconds
Finished 'from_pb' in 0.8524 seconds
Finished 'from_pb' in 0.3828 seconds
Finished 'from_pb' in 0.353 seconds
Finished 'from_pb' in 0.4222 seconds
Finished 'from_pb' in 0.3854 seconds
Finished 'from_pb' in 0.2875 seconds
Finished 'from_pb' in 0.3128 seconds
Finished 'from_pb' in 0.2053 seconds
Finished 'from_pb' in 0.3051 seconds
Finished 'from_pb' in 0.0501 seconds
Finished 'from_pb' in 0.0419 seconds
Finished 'from_pb' in 0.0359 seconds
Fin

Compute data on each tree

In [4]:
# subtypes = ['H3N2', 'H1N1', 'H1N1pdm', 'H7N9', 'H5N1', 'H2N2', 'H9N2']
muts_per_branch_cutoff = 4
for subtype in subtypes:
    for segment in segments:
        
        if segment not in trees[subtype]:
            continue

        if segment != 'HA':
            continue

        print('#--------------------------')
        print(subtype, segment)
        
        tree = trees[subtype][segment]
        nodes = tree.depth_first_expansion()
        print('Number of branches in tree:', len(nodes))
        muts_per_branch = Counter()
        passing_muts = 0
        total_muts = 0
        for (i, node) in enumerate(nodes):
            n_muts = len(node.mutations)
            total_muts += n_muts
            muts_per_branch += Counter([n_muts])
            if n_muts <= muts_per_branch_cutoff:
                passing_muts += n_muts

        print('Total mutations', total_muts)
        print('Passing mutations', passing_muts)

        df = pd.DataFrame(muts_per_branch.items(), columns=['muts_per_branch', 'n_branches'])
        df.sort_values('muts_per_branch', inplace=True)

        total_branches = df['n_branches'].sum()
        short_branches = df[df['muts_per_branch'] <= muts_per_branch_cutoff]['n_branches'].sum()
        # print total branches
        print('Fraction of short branches:', short_branches / total_branches)

        display(df.head(n=10))

        print('\n')

#--------------------------
H1N1 HA
Number of branches in tree: 174065
Total mutations 126865
Passing mutations 108044
Fraction of short branches: 0.9885445092350559


,muts_per_branch,n_branches
0,0,96117
1,1,53194
2,2,15449
7,3,5292
5,4,2019
14,5,864
3,6,351
15,7,187
4,8,114
16,9,84




#--------------------------
H3N2 HA
Number of branches in tree: 230969
Total mutations 171931
Passing mutations 133492
Fraction of short branches: 0.9840974329888427


,muts_per_branch,n_branches
0,0,135732
4,1,62478
3,2,19142
1,3,7046
5,4,2898
8,5,1206
2,6,634
11,7,374
16,8,235
17,9,193




#--------------------------
H5N1 HA
Number of branches in tree: 44538
Total mutations 55311
Passing mutations 31174
Fraction of short branches: 0.9412636400377206


,muts_per_branch,n_branches
0,0,23222
3,1,10963
1,2,4240
4,3,2257
6,4,1240
2,5,732
12,6,526
5,7,314
9,8,233
10,9,156




#--------------------------
H7N9 HA
Number of branches in tree: 4034
Total mutations 7702
Passing mutations 2984
Fraction of short branches: 0.91398116013882


,muts_per_branch,n_branches
0,0,1870
1,1,1079
5,2,429
2,3,189
4,4,120
3,5,75
9,6,41
8,7,37
7,8,27
13,9,24




#--------------------------
H9N2 HA
Number of branches in tree: 23057
Total mutations 87510
Passing mutations 20519
Fraction of short branches: 0.7637593789304766


,muts_per_branch,n_branches
0,0,7096
14,1,4826
8,2,2640
11,3,1779
17,4,1269
18,5,911
15,6,718
1,7,608
3,8,443
20,9,373


Compute data on each tree

In [3]:
subtypes = ['H3N2', 'H1N1', 'H1N1pdm', 'H7N9', 'H5N1', 'H2N2', 'H9N2']
muts_per_branch_cutoff = 8
for subtype in subtypes:
    print('#--------------------------')
    print('Subtype', subtype)
    
    tree = trees[subtype]
    nodes = tree.depth_first_expansion()
    print('Number of branches in tree:', len(nodes))
    muts_per_branch = Counter()
    passing_muts = 0
    total_muts = 0
    for (i, node) in enumerate(nodes):
        n_muts = len(node.mutations)
        total_muts += n_muts
        muts_per_branch += Counter([n_muts])
        if n_muts <= muts_per_branch_cutoff:
            passing_muts += n_muts

    print('Total mutations', total_muts)
    print('Passing mutations', passing_muts)

    df = pd.DataFrame(muts_per_branch.items(), columns=['muts_per_branch', 'n_branches'])
    df.sort_values('muts_per_branch', inplace=True)

    total_branches = df['n_branches'].sum()
    short_branches = df[df['muts_per_branch'] <= muts_per_branch_cutoff]['n_branches'].sum()
    # print total branches
    print('Fraction of short branches:', short_branches / total_branches)

    display(df.head(n=10))

    print('\n')

#--------------------------
Subtype H3N2
Number of branches in tree: 103143
Total mutations 130334
Passing mutations 89501
Fraction of short branches: 0.9785734368788963


,muts_per_branch,n_branches
0,0,52293
2,1,29023
1,2,9945
4,3,4371
5,4,2283
6,5,1223
7,6,775
3,7,582
9,8,438
10,9,344




#--------------------------
Subtype H1N1
Number of branches in tree: 49568
Total mutations 110009
Passing mutations 61673
Fraction of short branches: 0.9493423176242737


,muts_per_branch,n_branches
0,0,19196
3,1,13612
2,2,6056
1,3,3240
20,4,1878
12,5,1205
10,6,827
5,7,614
17,8,429
13,9,370




#--------------------------
Subtype H1N1pdm
Number of branches in tree: 81210
Total mutations 125822
Passing mutations 82449
Fraction of short branches: 0.9708287156754094


...

,muts_per_branch,n_branches
0,0,17412
6,1,8519
1,2,3742
5,3,2057
4,4,1340
7,5,802
3,6,646
20,7,402
12,8,304
8,9,246




#--------------------------
Subtype H2N2
Number of branches in tree: 1853
Total mutations 10002
Passing mutations 3224
Fraction of short branches: 0.8267674042093902


,muts_per_branch,n_branches
0,0,407
7,1,397
8,2,224
3,3,149
4,4,113
10,5,93
12,6,65
5,7,47
27,8,37
2,9,35




#--------------------------
Subtype H9N2
Number of branches in tree: 27215
Total mutations 99196
Passing mutations 44143
Fraction of short branches: 0.8840345397758589


,muts_per_branch,n_branches
0,0,8317
1,1,5825
5,2,3175
4,3,2077
9,4,1483
6,5,1070
8,6,871
2,7,699
13,8,542
3,9,454
